In [1]:
import pandas as pd

ml_table = pd.read_csv("../processedData/ml_table_labeled.csv")
print(ml_table.shape)

(99441, 18)


In [2]:
ml_table["order_purchase_timestamp"] = pd.to_datetime(ml_table["order_purchase_timestamp"])

# ترتيب الجدول زمنيًا من الأقدم للأحدث
ml_table_sorted = ml_table.sort_values("order_purchase_timestamp").reset_index(drop=True)

# حساب حدود التقسيم
n = len(ml_table_sorted)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = ml_table_sorted.iloc[:train_end]
val_df = ml_table_sorted.iloc[train_end:val_end]
test_df = ml_table_sorted.iloc[val_end:]

print("Train:", train_df.shape, train_df["order_purchase_timestamp"].min(), "to", train_df["order_purchase_timestamp"].max())
print("Val:", val_df.shape, val_df["order_purchase_timestamp"].min(), "to", val_df["order_purchase_timestamp"].max())
print("Test:", test_df.shape, test_df["order_purchase_timestamp"].min(), "to", test_df["order_purchase_timestamp"].max())

Train: (69608, 18) 2016-09-04 21:15:19 to 2018-04-13 21:50:49
Val: (14916, 18) 2018-04-13 21:53:34 to 2018-06-20 16:28:00
Test: (14917, 18) 2018-06-20 16:28:15 to 2018-10-17 17:30:18


In [3]:
print("Train:")
print(train_df["is_late"].value_counts(normalize=True))
print()
print("Validation:")
print(val_df["is_late"].value_counts(normalize=True))
print()
print("Test:")
print(test_df["is_late"].value_counts(normalize=True))

Train:
is_late
0    0.912596
1    0.087404
Name: proportion, dtype: float64

Validation:
is_late
0    0.947305
1    0.052695
Name: proportion, dtype: float64

Test:
is_late
0    0.935845
1    0.064155
Name: proportion, dtype: float64


### Label Balance Across Splits

| Split | Late (%) |
|---|---|
| Train | 8.74% |
| Validation | 5.27% |
| Test | 6.42% |

**Observation:** Train has a noticeably higher late-delivery rate than 
Validation/Test. This is expected with a time-based split — earlier 
orders (Train) come from when the company was likely still refining 
its delivery estimation and logistics, while later orders (Val/Test) 
reflect an improved, more mature operation. This is a real pattern in 
the data, not a bug in the split — kept as-is to preserve realistic 
temporal evaluation.

In [4]:
train_df.to_csv("../processedData/train.csv", index=False)
val_df.to_csv("../processedData/val.csv", index=False)
test_df.to_csv("../processedData/test.csv", index=False)
print("Saved successfully!")

Saved successfully!
